# Qwen2.5-VL 7B Instruct Installation & Test

## Overview

This notebook tests the Qwen2.5-VL-7B-Instruct model on extracting fields from a loan application form image.

**Model Info:**
- Model: Qwen/Qwen2.5-VL-7B-Instruct
- Quantization: 4-bit (NF4) using BitsAndBytes
- Task: Extract structured fields from a form image and output JSON

**Prerequisites:**
- GPU runtime (T4 or higher recommended)
- Image file: `sample_form.png` in the current working directory

**Output:**
- Model-generated JSON extraction result
- Time and memory usage metrics for later comparison

In [1]:
# Installation
import subprocess
import sys
from typing import Optional

def install_package(package: str, quiet: bool = True) -> bool:
    """
    Install a Python package.

    Args:
        package: Package name (supports git+url format)
        quiet: Whether to install quietly

    Returns:
        True if installation succeeded, False otherwise
    """
    try:
        cmd = [sys.executable, "-m", "pip", "install"]
        if quiet:
            cmd.append("-q")
        cmd.append(package)
        subprocess.check_call(cmd)
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to install: {package}\nError: {e}")
        return False

# Required packages – transformers must be installed from source
packages = [
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "bitsandbytes",
    "qwen-vl-utils",
    "pillow",
]

for pkg in packages:
    print(f"📦 Installing: {pkg}")
    install_package(pkg)

print("✅ All dependencies installed successfully")

📦 Installing: git+https://github.com/huggingface/transformers
📦 Installing: accelerate
📦 Installing: bitsandbytes
📦 Installing: qwen-vl-utils
📦 Installing: pillow
✅ All dependencies installed successfully


In [2]:
# Imports
import json
import time
import re
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import torch
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info
from PIL import Image

def check_device() -> str:
    """Check and return the available device."""
    if torch.cuda.is_available():
        print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"   GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        return "cuda"
    else:
        print("⚠️  GPU not available – using CPU (slow)")
        return "cpu"

device: str = check_device()

✅ GPU available: Tesla T4
   GPU memory: 15.64 GB


In [ ]:
# Mount Google Drive and copy sample image
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/Fairform/sample_form.png .   # adjust path as needed

# Verify image
from pathlib import Path
from PIL import Image

IMAGE_PATH = "sample_form.png"

def verify_image(image_path: str) -> bool:
    path = Path(image_path)
    if not path.exists():
        print(f"❌ Image not found: {image_path}")
        return False
    with Image.open(path) as img:
        print(f"✅ Image verified: {img.size}, {img.format}")
        return True

if not verify_image(IMAGE_PATH):
    raise FileNotFoundError("sample_form.png is required")

In [ ]:
# Load model and processor
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# 8‑bit quantization (stable)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✅ Model and processor loaded")
print(f"Model device: {next(model.parameters()).device}")

In [22]:
# Build Messages
def build_messages(
    image_path: str,
    instruction: Optional[str] = None,
) -> List[Dict[str, Any]]:
    """
    Build the multimodal message format for Qwen2.5-VL.

    Args:
        image_path: Path to the image
        instruction: Custom instruction (optional)

    Returns:
        List of messages
    """
    if instruction is None:
        instruction = (
            "Extract all fields from this loan application form. "
            "Output as a valid JSON object with field names as keys "
            "and extracted values as strings."
        )

    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": instruction},
            ],
        }
    ]

messages: List[Dict[str, Any]] = build_messages(IMAGE_PATH)
print("✅ Messages built")
print(f"   - Instruction: {messages[0]['content'][1]['text'][:80]}...")

✅ Messages built
   - Instruction: Extract all fields from this loan application form. Output as a valid JSON objec...


In [ ]:
# Prepare Inputs
def prepare_inputs(
    messages: List[Dict[str, Any]],
    processor: AutoProcessor,
    device:str ="cuda:0"
) -> Dict[str, torch.Tensor]:
    """
    Prepare model inputs from messages.

    Args:
        messages: Multimodal message list
        processor: The processor
        device: Target device

    Returns:
        Dictionary of input tensors
    """
    text: str = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    return inputs.to(device)

inputs: Dict[str, torch.Tensor] = prepare_inputs(messages, processor, device)
print("✅ Input preparation complete")
print(f"   - Input shape: {inputs['input_ids'].shape}")

✅ Input preparation complete
   - Input shape: torch.Size([1, 747])


In [ ]:
# Generate Response
def generate_response(
    model: Qwen2VLForConditionalGeneration,
    inputs: Dict[str, torch.Tensor],
    processor: AutoProcessor,
    max_new_tokens: int = 512,
    temperature: float = 0.1,
    do_sample: bool = False,
) -> Tuple[str, float]:

    
    """
    Run model inference and return generated text.

    Args:
        model: The loaded model
        inputs: Input tensors
        processor: The processor
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (low = deterministic)
        do_sample: Whether to sample

    Returns:
        Tuple of (generated_text, inference_time_seconds)
    """
    print("🔄 Generating response...")
    start_time: float = time.time()

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    # Trim input tokens from the output
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]

    output_text: str = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    elapsed: float = time.time() - start_time
    print(f"✅ Generation complete (took {elapsed:.2f} seconds)")

    return output_text, elapsed

output_text, inference_time = generate_response(model, inputs, processor)

🔄 Generating response...


AssertionError: 

In [ ]:
# Parse & Display Result
def parse_and_display_result(
    output_text: str,
    inference_time: float,
) -> Dict[str, Any]:
    """
    Parse and display the model output.

    Args:
        output_text: Raw output from the model
        inference_time: Time taken for inference

    Returns:
        Parsed JSON dictionary if parsable, otherwise raw text wrapped
    """
    print("\n" + "=" * 60)
    print("📋 Extraction Result")
    print("=" * 60)
    print(f"\n⏱️  Inference time: {inference_time:.2f} seconds")
    print(f"📝 Output length: {len(output_text)} characters\n")

    print("-" * 60)
    print("Raw output:")
    print("-" * 60)
    print(output_text)
    print("-" * 60)

    result: Dict[str, Any] = {}
    try:
        # Try to extract JSON if wrapped in markdown code block
        json_match = re.search(r'```json\s*([\s\S]*?)\s*```', output_text)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_str = output_text

        result = json.loads(json_str)
        print("\n✅ Successfully parsed as JSON:")
        print(json.dumps(result, indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print("\n⚠️  Output is not valid JSON")
        result = {"raw_output": output_text}

    return result

result = parse_and_display_result(output_text, inference_time)

In [ ]:
# Save Results
def save_results(
    output_text: str,
    result: Dict[str, Any],
    inference_time: float,
    output_path: str = "qwen_extraction_result.json",
) -> None:
    """
    Save extraction results to a JSON file for later comparison.

    Args:
        output_text: Raw output text
        result: Parsed result
        inference_time: Inference time in seconds
        output_path: Output file path
    """
    save_data = {
        "model": MODEL_ID,
        "image": IMAGE_PATH,
        "inference_time_seconds": inference_time,
        "raw_output": output_text,
        "parsed_result": result,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(save_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Results saved to: {output_path}")

save_results(output_text, result, inference_time)

In [ ]:
# Memory Stats
def get_memory_stats() -> Dict[str, float]:
    """
    Get current GPU memory usage statistics.

    Returns:
        Dictionary with memory stats in GB
    """
    if not torch.cuda.is_available():
        return {"error": "CUDA not available"}

    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9

    return {
        "allocated_gb": allocated,
        "reserved_gb": reserved,
        "max_allocated_gb": torch.cuda.max_memory_allocated() / 1e9,
    }

memory_stats: Dict[str, float] = get_memory_stats()
print("\n📊 GPU Memory Usage:")
print(f"   - Allocated: {memory_stats.get('allocated_gb', 0):.2f} GB")
print(f"   - Reserved: {memory_stats.get('reserved_gb', 0):.2f} GB")
print(f"   - Peak allocated: {memory_stats.get('max_allocated_gb', 0):.2f} GB")

In [ ]:
# Cleanup 
def cleanup() -> None:
    """Clear GPU cache to free memory."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("✅ GPU cache cleared")

# Uncomment the line below to run cleanup
# cleanup()